# The Same Dice Game, Eleven Ways: From Python to Machine Code, and Beyond

One tiny program, written at every level of the stack, so you can see what changes as you move closer to the hardware and what stays the same.

**The game:** you and the computer each roll two dice per round. The higher total wins the round (ties score nothing). Five rounds, then a winner, then an optional rematch.

| # | Version | Era | What it is |
|---|---------|-----|-----------|
| 1 | Python | 1991 | Readable and built-in random numbers; the baseline for everything below |
| 2 | Fortran 90 | 1990s | Modern free-form Fortran with a built-in random number generator |
| 3 | Fortran 77 | 1977 | Fixed-form, column-sensitive Fortran with `GO TO` and numbered labels |
| 4 | BASIC | 1964+ | Line-numbered, interpreted, the language of early home computers |
| 5 | C | 1970s | Compiled, close to the hardware, still everywhere today |
| 6 | x86-64 assembly | 1950s idea | One text instruction per CPU instruction, no libc, raw Linux syscalls |
| 7 | Machine code | the CPU's own language | The same program as raw bytes (hex), including a hand-built ELF header |
| 8 | Brainfuck | 1993 | Eight commands and a tape of bytes |
| 9 | Chef | 2002 | Programs that read like cooking recipes |
| 10 | Piet | 2001 | Programs that are abstract paintings |
| 11 | Whitespace | 2003 | Programs written only in spaces, tabs and newlines |

Versions 1 to 7 are a ladder from a high-level language down to the machine. Versions 8 to 11 are **esoteric languages**: they run the same game on strange models of computation, to show how little a language needs.

**How to use this notebook:** run the cells top to bottom. Each section writes a source file, builds it, and runs it.

> **A note on scope.** The esoteric versions simplify the game where the language makes it impractical: Brainfuck and Chef play one game and stop (no rematch), and Piet plays a single round. Each section says exactly what it does.

> **A note on input.** Colab's `!` commands can't read your keyboard, so each run pipes in scripted input instead: one Enter per round, then `n` to decline a rematch. To play interactively, run the programs in a Colab terminal or on your own machine.

## The abstraction ladder

```
 Fortran 90 / Fortran 77 / BASIC / C     <- you describe WHAT to do
            |  compiler or interpreter
            v
        assembly language                <- one readable line per CPU instruction
            |  assembler
            v
         machine code                    <- the bytes the CPU actually executes
            |  CPU decoder / microcode
            v
        logic gates and transistors
```

Everything above machine code is a convenience for humans. The CPU only ever sees the bytes at the bottom.

## Before we start: how does a computer run a program?

If you have never programmed, a few words come up constantly in this notebook. Here they are in plain language. Come back to this section any time.

**Source code** is the text a person writes, like `print("hi")`. A computer's processor (the CPU) can't run it directly. The CPU only understands **machine code**: long lists of numbers, each one a tiny instruction such as "add these two numbers" or "jump to another instruction".

So every program has to be **translated** from source code into something the CPU can run. There are a few ways to do that, and which one a language uses explains many of the differences you'll see below.

| Word | What it is | Everyday comparison |
|------|-----------|---------------------|
| **Compiler** | A program that translates *all* of your source code into machine code *ahead of time* and saves the result as a new file you can run any time | Translating a whole book once, then handing out copies |
| **Interpreter** | A program that reads your source code and *does what it says as it goes*, without making a separate translated file | A live interpreter at a meeting, translating sentence by sentence |
| **Assembler** | A small compiler for *assembly language*, which is machine code written as short readable words (`add`, `mov`) instead of numbers | Writing each number out in words |
| **Linker** | Joins separate pieces of translated code, and shared code from libraries, into one runnable file | Binding chapters into one book |
| **Executable** | The finished file the operating system can run directly | The printed book |

A few more ideas:
- **Library:** code someone else wrote that you can reuse, such as the code that prints text on the screen.
- **Operating system (OS):** the program that manages the computer (Colab runs Linux). Programs ask it for things like "print this" or "read a key" through **system calls**.
- **Stack:** a pile where you only add or remove at the top, like a stack of plates. Several of the languages later work this way.
- **Pseudo-random numbers:** a computer can't truly roll dice. It uses a formula that turns one number into the next in a way that looks random. The starting number is the **seed**: the same seed gives the same "random" rolls. Some versions here ask you to type a seed, and others take one from the clock.

### How to read the cells in this notebook

- A cell starting with `%%writefile name` saves everything below that line as a file called `name` on Colab's computer. It doesn't run anything.
- A line starting with `!` runs a command, as if you typed it in a terminal. For example, `!gcc dice_game.c -o dice_c` runs the C compiler.
- `printf '\n\n\n' | ./program` feeds text into a program as if you had typed it, where `\n` means "press Enter". We do this because Colab can't pause for typing while a command runs. The `|` is called a *pipe*: it sends one command's output into the next command's input.

## 0. Setup

This installs the tools we need:

- `gfortran` compiles both Fortran versions
- `bwbasic` is a BASIC interpreter (Bywater BASIC)
- `gcc` (C compiler), `as` (assembler) and `ld` (linker) usually come preinstalled on Colab
- `xxd` converts between hex text and binary files
- Pillow and numpy (preinstalled on Colab) are used to draw and read the Piet image

The last line prints where each tool lives, so you can see if anything is missing.

In [ ]:
!apt-get install -y -qq gfortran bwbasic xxd > /dev/null 2>&1
!pip install -q pillow numpy
!uname -m
!which python3 gfortran bwbasic gcc as ld xxd

`uname -m` should print `x86_64`. The assembly and machine-code sections are written for x86-64 Linux, which is what Colab runs. They won't run on ARM machines (including Apple Silicon) or on Windows.

## 1. Python

Python (Guido van Rossum, 1991) is the language most people meet first, and it's the easiest way to say what the game does. Everything below is the same game with more of the work done by hand.

Things to notice:
- `random.randint(1, 6)` rolls a die in one line. There is nothing to seed and no generator to write.
- Indentation is the syntax: no braces and no `end` keywords.
- **f-strings** put variables straight into text, like `f"You rolled: {p1} + {p2} = {p_total}"`.
- `input()` waits for Enter, so unlike the other versions this one can be played **interactively right in Colab**.
- Python counts from 0, like C and unlike Fortran. We don't index any lists here, but it's the first thing you'd hit.

**What we get:** no build step. Python is an **interpreter**: when you run `python3 dice_game.py`, it reads your text file and carries out the instructions as it goes. (Internally it first turns the file into a compact form called *bytecode*, which a built-in engine then runs.) So the only file involved is the source file itself. That makes Python quick to try and change, at the cost of some speed, because the translating happens every time you run it.

In [ ]:
%%writefile dice_game.py
"""Dice game: you vs. the computer.
Each round both sides roll two dice; the highest total wins the round."""

import random

N_ROUNDS = 5


def roll_die():
    return random.randint(1, 6)


def play_game():
    p_wins = c_wins = 0

    print("=== Dice Game: You vs. Computer ===")
    print(f"Best of {N_ROUNDS} rounds; highest total wins.")

    for round_no in range(1, N_ROUNDS + 1):
        print(f"\nRound {round_no}")
        print("Press Enter to roll...")
        input()

        p1, p2 = roll_die(), roll_die()
        c1, c2 = roll_die(), roll_die()
        p_total, c_total = p1 + p2, c1 + c2

        print(f"You rolled:      {p1} + {p2} = {p_total}")
        print(f"Computer rolled: {c1} + {c2} = {c_total}")

        if p_total > c_total:
            p_wins += 1
            print("You win this round!")
        elif c_total > p_total:
            c_wins += 1
            print("Computer wins this round.")
        else:
            print("Tie -- no points.")

        print(f"Score: You {p_wins} - Computer {c_wins}")

    print("\n=== Final Result ===")
    if p_wins > c_wins:
        print("You win the game!")
    elif c_wins > p_wins:
        print("The computer wins the game.")
    else:
        print("The game is a draw.")


def main():
    while True:
        play_game()
        print("\nPlay again? (y/n)")
        if input().strip().lower() != "y":
            break
    print("Thanks for playing!")


if __name__ == "__main__":
    try:
        main()
    except EOFError:
        print("\nThanks for playing!")

In [ ]:
!printf '\n\n\n\n\nn\n' | python3 dice_game.py

The cell above feeds in scripted input. To play it for real, run the cell below and type into the box that appears: press Enter for each roll, then `y` or `n` at the end.

In [ ]:
%run dice_game.py

## 2. Fortran 90

**Fortran** (FORmula TRANslation, IBM, 1957) was the first widely used high-level language, built for scientists and engineers. Fortran 90 modernized it with free-form layout (no column rules), `do ... end do` loops, and `if / else if` blocks.

Things to notice in the code:
- `random_number` and `random_seed` are built into the language, so rolling a die is one line.
- Arrays and loops count from 1 by default in Fortran.
- A small function (`roll_die`) lives in a `contains` section inside the program.

**What we get:** a native executable, `dice_f90`, produced by the compiler.

In [ ]:
%%writefile dice_game.f90
program dice_game
  implicit none

  integer, parameter :: n_rounds = 5
  integer :: round, p1, p2, c1, c2, p_total, c_total
  integer :: p_wins, c_wins, ios
  character(len=80) :: line
  character(len=1) :: again

  call random_seed()

  do
    p_wins = 0
    c_wins = 0

    print '(a)', '=== Dice Game: You vs. Computer ==='
    print '(a,i0,a)', 'Highest total of two dice wins each round (', n_rounds, ' rounds).'

    do round = 1, n_rounds
      print '(/,a,i0)', 'Round ', round
      print '(a)', 'Press Enter to roll...'
      read (*, '(a)', iostat=ios) line
      if (ios /= 0) stop

      p1 = roll_die()
      p2 = roll_die()
      c1 = roll_die()
      c2 = roll_die()
      p_total = p1 + p2
      c_total = c1 + c2

      print '(a,i0,a,i0,a,i0)', 'You rolled:      ', p1, ' + ', p2, ' = ', p_total
      print '(a,i0,a,i0,a,i0)', 'Computer rolled: ', c1, ' + ', c2, ' = ', c_total

      if (p_total > c_total) then
        p_wins = p_wins + 1
        print '(a)', 'You win this round!'
      else if (c_total > p_total) then
        c_wins = c_wins + 1
        print '(a)', 'Computer wins this round.'
      else
        print '(a)', 'Tie -- no points.'
      end if

      print '(a,i0,a,i0)', 'Score: You ', p_wins, ' - Computer ', c_wins
    end do

    print '(/,a)', '=== Final Result ==='
    if (p_wins > c_wins) then
      print '(a)', 'You win the game!'
    else if (c_wins > p_wins) then
      print '(a)', 'The computer wins the game.'
    else
      print '(a)', 'The game is a draw.'
    end if

    print '(/,a)', 'Play again? (y/n)'
    read (*, '(a)', iostat=ios) again
    if (ios /= 0) exit
    if (again /= 'y' .and. again /= 'Y') exit
    print *
  end do

  print '(a)', 'Thanks for playing!'

contains

  integer function roll_die()
    real :: r
    call random_number(r)
    roll_die = 1 + int(r * 6.0)
  end function roll_die

end program dice_game

### Step 1: compile. Step 2: run.

`gfortran` is a **compiler**. The next cell hands it your text file and asks for an executable called `dice_f90`. It reads the whole program, checks it for mistakes, and translates it into machine code for this computer's processor. Nothing runs yet, and if the text has an error you'll see an error message here instead of a program.

The cell after that is step 2: running the finished executable. The difference from Python is *when* the translation happens: once, ahead of time, instead of every time you run the program.

In [ ]:
!gfortran dice_game.f90 -o dice_f90

In [ ]:
!printf '\n\n\n\n\nn\n' | ./dice_f90

## 3. Fortran 77

Fortran 77 is the version much of the world's scientific code was written in. It is **fixed-form**: the column you type in matters.

| Columns | Meaning |
|---------|---------|
| 1 | `C` here makes the whole line a comment |
| 1-5 | Statement label (a number like `10` or `100`) |
| 6 | Any character here means "continuation of the previous line" |
| 7-72 | The actual statement |

What's different from Fortran 90:
- No `END DO`: loops end on a labeled `CONTINUE`, e.g. `DO 50 ROUND = 1, NROUND` ... `50 CONTINUE`.
- No `CONTAINS`, so the die roller is a separate `FUNCTION` after the main program.
- Output layout uses numbered `FORMAT` statements.
- **No standard random number routine**, so the program carries its own generator (Park-Miller, using Schrage's trick so it never overflows a 32-bit integer). With no portable clock either, it asks you for a **seed number**. The same seed gives the same game.

**What we get:** another native executable, `dice_f77`. Same compiler, older dialect.

In [ ]:
%%writefile dice_game.f
C     DICE GAME: YOU VS. THE COMPUTER (FORTRAN 77, FIXED FORM)
C     EACH ROUND BOTH SIDES ROLL TWO DICE; HIGHEST TOTAL WINS.
      PROGRAM DICE
      INTEGER NROUND, ROUND, P1, P2, C1, C2, PTOT, CTOT
      INTEGER PWINS, CWINS, ISEED, IROLL
      CHARACTER*1 AGAIN
      CHARACTER*80 LINE
      PARAMETER (NROUND = 5)

      WRITE(*,'(A)') ' Enter a whole number to seed the dice:'
      READ(*,*,ERR=900,END=900) ISEED
      ISEED = MOD(ABS(ISEED), 2147483646) + 1

   10 CONTINUE
      PWINS = 0
      CWINS = 0
      WRITE(*,100)
      WRITE(*,110) NROUND

      DO 50 ROUND = 1, NROUND
         WRITE(*,120) ROUND
         WRITE(*,'(A)') ' Press Enter to roll...'
         READ(*,'(A)',END=900) LINE

         P1 = IROLL(ISEED)
         P2 = IROLL(ISEED)
         C1 = IROLL(ISEED)
         C2 = IROLL(ISEED)
         PTOT = P1 + P2
         CTOT = C1 + C2

         WRITE(*,130) P1, P2, PTOT
         WRITE(*,140) C1, C2, CTOT

         IF (PTOT .GT. CTOT) THEN
            PWINS = PWINS + 1
            WRITE(*,'(A)') ' You win this round!'
         ELSE IF (CTOT .GT. PTOT) THEN
            CWINS = CWINS + 1
            WRITE(*,'(A)') ' Computer wins this round.'
         ELSE
            WRITE(*,'(A)') ' Tie -- no points.'
         END IF

         WRITE(*,150) PWINS, CWINS
   50 CONTINUE

      WRITE(*,160)
      IF (PWINS .GT. CWINS) THEN
         WRITE(*,'(A)') ' You win the game!'
      ELSE IF (CWINS .GT. PWINS) THEN
         WRITE(*,'(A)') ' The computer wins the game.'
      ELSE
         WRITE(*,'(A)') ' The game is a draw.'
      END IF

      WRITE(*,'(/A)') ' Play again? (y/n)'
      READ(*,'(A)',END=900) AGAIN
      IF (AGAIN .EQ. 'y' .OR. AGAIN .EQ. 'Y') GO TO 10

  900 WRITE(*,'(A)') ' Thanks for playing!'
      STOP

  100 FORMAT(' === Dice Game: You vs. Computer ===')
  110 FORMAT(' Best of ',I2,' rounds; highest total wins.')
  120 FORMAT(/' Round ',I2)
  130 FORMAT(' You rolled:      ',I1,' + ',I1,' = ',I2)
  140 FORMAT(' Computer rolled: ',I1,' + ',I1,' = ',I2)
  150 FORMAT(' Score: You ',I2,' - Computer ',I2)
  160 FORMAT(/' === Final Result ===')
      END

C     ROLL ONE DIE (1-6). F77 HAS NO STANDARD RANDOM NUMBER ROUTINE,
C     SO THIS USES THE PARK-MILLER GENERATOR (SCHRAGE'S METHOD, SO
C     IT NEVER OVERFLOWS A 32-BIT INTEGER). ISEED IS UPDATED IN PLACE.
      INTEGER FUNCTION IROLL(ISEED)
      INTEGER ISEED, K
      K = ISEED / 127773
      ISEED = 16807 * (ISEED - K * 127773) - 2836 * K
      IF (ISEED .LT. 0) ISEED = ISEED + 2147483647
      IROLL = 1 + INT(6.0D0 * DBLE(ISEED) / 2147483647.0D0)
      RETURN
      END

This is the same two steps as before: compile with `gfortran`, then run. Modern compilers still accept old-style Fortran, because it is a valid dialect of the language.

**Why does it ask for a seed?** A computer can't truly roll dice. It uses a formula that produces numbers that look random, and the formula needs a starting number, the **seed**. Fortran 90 has a generator built in that picks its own seed. Fortran 77 doesn't, so this program contains its own formula and asks you for the seed. Same seed, same game.

In [ ]:
!gfortran dice_game.f -o dice_f77

In [ ]:
!printf '42\n\n\n\n\n\nn\n' | ./dice_f77

Change the `42` to any whole number and you'll get a different sequence of rolls.

## 4. BASIC

BASIC (Dartmouth, 1964) was designed so students could program without being experts. Programs are **numbered lines**, run in order unless you jump with `GOTO` or `GOSUB`, and are usually **interpreted**: a program reads each line and executes it directly, with no separate compile step.

Things to notice:
- `RND(1)` gives a random number between 0 and 1; `INT(RND(1) * 6) + 1` turns that into a die roll.
- `GOSUB 900` calls a tiny subroutine at line 900 that rolls one die.
- Several statements can share a line, separated by `:`.
- Like the Fortran 77 version, it asks for a seed.

**What we get:** no executable at all. The interpreter (`bwbasic`) reads the text file each time you run it.

> This section was written for the widely compatible classic dialect and I haven't been able to test it under `bwbasic`. If a line misbehaves, it's most likely the multi-statement `IF ... THEN` lines (240, 250, 320, 330).

In [ ]:
%%writefile dice_game.bas
10 REM DICE GAME: YOU VS. THE COMPUTER
20 REM EACH ROUND BOTH SIDES ROLL TWO DICE; HIGHEST TOTAL WINS.
30 N = 5
40 INPUT "Enter a whole number to seed the dice"; S
50 RANDOMIZE S
100 PW = 0 : CW = 0
110 PRINT "=== Dice Game: You vs. Computer ==="
120 PRINT "Best of "; N; " rounds; highest total wins."
130 FOR R = 1 TO N
140 PRINT
150 PRINT "Round "; R
160 INPUT "Press Enter to roll..."; A$
170 GOSUB 900 : P1 = D
180 GOSUB 900 : P2 = D
190 GOSUB 900 : C1 = D
200 GOSUB 900 : C2 = D
210 PT = P1 + P2 : CT = C1 + C2
220 PRINT "You rolled:      "; P1; "+"; P2; "="; PT
230 PRINT "Computer rolled: "; C1; "+"; C2; "="; CT
240 IF PT > CT THEN PW = PW + 1 : PRINT "You win this round!" : GOTO 280
250 IF CT > PT THEN CW = CW + 1 : PRINT "Computer wins this round." : GOTO 280
260 PRINT "Tie -- no points."
280 PRINT "Score: You"; PW; "- Computer"; CW
290 NEXT R
300 PRINT
310 PRINT "=== Final Result ==="
320 IF PW > CW THEN PRINT "You win the game!" : GOTO 350
330 IF CW > PW THEN PRINT "The computer wins the game." : GOTO 350
340 PRINT "The game is a draw."
350 PRINT
360 INPUT "Play again? (y/n)"; A$
370 IF A$ = "y" OR A$ = "Y" THEN 100
380 PRINT "Thanks for playing!"
390 END
900 REM ROLL ONE DIE (1-6) INTO D
910 D = INT(RND(1) * 6) + 1
920 RETURN

### No build step: the interpreter does the work

Unlike the Fortran versions, there is nothing to compile here. `bwbasic` is an **interpreter**: a program that reads the BASIC file and does what each numbered line says, in order. It never produces a separate executable file. That makes BASIC easy to experiment with (change a line, run again), but the interpreter has to read and understand every line each time the program runs.

In [ ]:
!printf '42\n\n\n\n\n\nn\n' | bwbasic dice_game.bas

## 5. C

C (Bell Labs, early 1970s) was created to write the Unix operating system. It is compiled straight to machine code, gives you direct control over memory, and has almost no runtime overhead. Operating systems, databases, and much of the software you use daily are written in it.

Things to notice:
- `rand()` and `srand()` come from the standard library, and we seed from the clock with `time(NULL)`.
- Unlike Fortran, **C counts from 0**. We don't index any arrays here, but it is the biggest day-to-day difference between the two.
- The code is longer than Fortran 90's only because of reading input carefully (`getchar`).

**What we get:** a native executable, `dice_c`, linked against the C standard library.

In [ ]:
%%writefile dice_game.c
/* Dice game: you vs. the computer.
   Each round both sides roll two dice; highest total wins. */

#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N_ROUNDS 5

static int roll_die(void)
{
    return (int)(rand() / (RAND_MAX + 1.0) * 6.0) + 1;
}

/* Wait for Enter. Returns 0 on end of input. */
static int wait_for_enter(void)
{
    int ch;
    do {
        ch = getchar();
        if (ch == EOF)
            return 0;
    } while (ch != '\n');
    return 1;
}

int main(void)
{
    int round, p1, p2, c1, c2, p_total, c_total;
    int p_wins, c_wins, again;

    srand((unsigned)time(NULL));

    do {
        p_wins = 0;
        c_wins = 0;

        printf("=== Dice Game: You vs. Computer ===\n");
        printf("Best of %d rounds; highest total wins.\n", N_ROUNDS);

        for (round = 1; round <= N_ROUNDS; round++) {
            printf("\nRound %d\n", round);
            printf("Press Enter to roll...\n");
            if (!wait_for_enter())
                return 0;

            p1 = roll_die();
            p2 = roll_die();
            c1 = roll_die();
            c2 = roll_die();
            p_total = p1 + p2;
            c_total = c1 + c2;

            printf("You rolled:      %d + %d = %d\n", p1, p2, p_total);
            printf("Computer rolled: %d + %d = %d\n", c1, c2, c_total);

            if (p_total > c_total) {
                p_wins++;
                printf("You win this round!\n");
            } else if (c_total > p_total) {
                c_wins++;
                printf("Computer wins this round.\n");
            } else {
                printf("Tie -- no points.\n");
            }

            printf("Score: You %d - Computer %d\n", p_wins, c_wins);
        }

        printf("\n=== Final Result ===\n");
        if (p_wins > c_wins)
            printf("You win the game!\n");
        else if (c_wins > p_wins)
            printf("The computer wins the game.\n");
        else
            printf("The game is a draw.\n");

        printf("\nPlay again? (y/n)\n");
        again = getchar();
        if (again == EOF)
            break;
        if (again != '\n')
            wait_for_enter();
    } while (again == 'y' || again == 'Y');

    printf("Thanks for playing!\n");
    return 0;
}

### Compiling and linking

`gcc` does two jobs in one command:
1. **Compile:** translate your text into machine code.
2. **Link:** attach the pieces your program borrows from the **standard library**, such as `printf` (printing text) and `rand` (random numbers). The `#include` lines at the top of the file tell the compiler which library features you plan to use. You didn't write `printf` yourself. The linker connects your program to the copy that ships with the system.

The result is the executable `dice_c`. The `-Wall -Wextra` options just ask the compiler to point out anything suspicious.

In [ ]:
!gcc -Wall -Wextra dice_game.c -o dice_c

In [ ]:
!printf '\n\n\n\n\nn\n' | ./dice_c

## 6. x86-64 Assembly

Assembly has **one line per CPU instruction**. There are no loops, no `if` blocks, and no variables, only registers (`rax`, `rbx`, ...), memory addresses, jumps, and compare instructions. This version uses GNU assembler syntax (Intel flavor) and **doesn't use the C library at all**. It talks to Linux directly through three system calls:

| Syscall | Number | Used for |
|---------|--------|----------|
| `read` | 0 | Waiting for Enter, reading `y`/`n` |
| `write` | 1 | Printing text |
| `exit` | 60 | Ending the program |

Everything else is written by hand:
- **Random numbers:** an xorshift64 generator (shifts and XORs on a 64-bit value), seeded from `rdtsc`, the CPU's cycle counter.
- **Printing numbers:** a loop that divides by 10 repeatedly and turns each remainder into an ASCII digit.
- **The game logic:** the same compare-and-jump pattern the compiled languages produce behind the scenes.

**What we get:** two steps instead of one. The *assembler* (`as`) turns the text into an object file, then the *linker* (`ld`) turns that into an executable.

In [ ]:
%%writefile dice_game.s
# Dice game: you vs. the computer (x86-64 Linux, GNU as, Intel syntax)
# Each round both sides roll two dice; highest total wins.
# No libc: raw Linux syscalls only (read=0, write=1, exit=60).
#
# Build:  as dice_game.s -o dice_game.o && ld dice_game.o -o dice_game
# Run:    ./dice_game

        .intel_syntax noprefix

        .equ N_ROUNDS, 5            # if you change this, also edit msg_rules

# ---------------------------------------------------------------- macros
.macro PRINT label
        mov     eax, 1              # sys_write
        mov     edi, 1              # stdout
        lea     rsi, [rip + \label]
        mov     edx, OFFSET \label\()_len
        syscall
.endm

# ---------------------------------------------------------------- data
        .section .rodata
msg_title:  .ascii "=== Dice Game: You vs. Computer ===\n"
        .set msg_title_len, . - msg_title
msg_rules:  .ascii "Best of 5 rounds; highest total wins.\n"
        .set msg_rules_len, . - msg_rules
msg_round:  .ascii "\nRound "
        .set msg_round_len, . - msg_round
msg_press:  .ascii "Press Enter to roll...\n"
        .set msg_press_len, . - msg_press
msg_you:    .ascii "You rolled:      "
        .set msg_you_len, . - msg_you
msg_cpu:    .ascii "Computer rolled: "
        .set msg_cpu_len, . - msg_cpu
msg_winp:   .ascii "You win this round!\n"
        .set msg_winp_len, . - msg_winp
msg_winc:   .ascii "Computer wins this round.\n"
        .set msg_winc_len, . - msg_winc
msg_tie:    .ascii "Tie -- no points.\n"
        .set msg_tie_len, . - msg_tie
msg_score1: .ascii "Score: You "
        .set msg_score1_len, . - msg_score1
msg_score2: .ascii " - Computer "
        .set msg_score2_len, . - msg_score2
msg_final:  .ascii "\n=== Final Result ===\n"
        .set msg_final_len, . - msg_final
msg_gwin:   .ascii "You win the game!\n"
        .set msg_gwin_len, . - msg_gwin
msg_closs:  .ascii "The computer wins the game.\n"
        .set msg_closs_len, . - msg_closs
msg_gdraw:  .ascii "The game is a draw.\n"
        .set msg_gdraw_len, . - msg_gdraw
msg_again:  .ascii "\nPlay again? (y/n)\n"
        .set msg_again_len, . - msg_again
msg_bye:    .ascii "Thanks for playing!\n"
        .set msg_bye_len, . - msg_bye
s_plus:     .ascii " + "
        .set s_plus_len, . - s_plus
s_eq:       .ascii " = "
        .set s_eq_len, . - s_eq
s_nl:       .ascii "\n"
        .set s_nl_len, . - s_nl

        .section .bss
seed:   .skip 8                     # xorshift64 state (never zero)
p1:     .skip 4
p2:     .skip 4
c1:     .skip 4
c2:     .skip 4
ptot:   .skip 4
ctot:   .skip 4
inbuf:  .skip 1
numbuf: .skip 24

# ---------------------------------------------------------------- code
        .text
        .globl _start

_start:
        rdtsc                       # seed from the CPU cycle counter
        shl     rdx, 32
        or      rax, rdx
        or      rax, 1              # xorshift state must be nonzero
        mov     [rip + seed], rax

play_game:
        xor     r13d, r13d          # r13 = player wins
        xor     r14d, r14d          # r14 = computer wins
        PRINT   msg_title
        PRINT   msg_rules
        mov     r12d, 1             # r12 = round number

round_loop:
        PRINT   msg_round
        mov     eax, r12d
        call    print_int
        PRINT   s_nl
        PRINT   msg_press
        call    wait_enter

        call    roll_die
        mov     [rip + p1], eax
        call    roll_die
        mov     [rip + p2], eax
        call    roll_die
        mov     [rip + c1], eax
        call    roll_die
        mov     [rip + c2], eax

        mov     eax, [rip + p1]
        add     eax, [rip + p2]
        mov     [rip + ptot], eax
        mov     eax, [rip + c1]
        add     eax, [rip + c2]
        mov     [rip + ctot], eax

        PRINT   msg_you
        mov     eax, [rip + p1]
        call    print_int
        PRINT   s_plus
        mov     eax, [rip + p2]
        call    print_int
        PRINT   s_eq
        mov     eax, [rip + ptot]
        call    print_int
        PRINT   s_nl

        PRINT   msg_cpu
        mov     eax, [rip + c1]
        call    print_int
        PRINT   s_plus
        mov     eax, [rip + c2]
        call    print_int
        PRINT   s_eq
        mov     eax, [rip + ctot]
        call    print_int
        PRINT   s_nl

        mov     eax, [rip + ptot]
        cmp     eax, [rip + ctot]
        jg      player_wins
        jl      cpu_wins
        PRINT   msg_tie
        jmp     show_score
player_wins:
        inc     r13d
        PRINT   msg_winp
        jmp     show_score
cpu_wins:
        inc     r14d
        PRINT   msg_winc

show_score:
        PRINT   msg_score1
        mov     eax, r13d
        call    print_int
        PRINT   msg_score2
        mov     eax, r14d
        call    print_int
        PRINT   s_nl

        inc     r12d
        cmp     r12d, N_ROUNDS
        jle     round_loop

        PRINT   msg_final
        cmp     r13d, r14d
        jg      game_won
        jl      game_lost
        PRINT   msg_gdraw
        jmp     ask_again
game_won:
        PRINT   msg_gwin
        jmp     ask_again
game_lost:
        PRINT   msg_closs

ask_again:
        PRINT   msg_again
        call    read_byte           # first character is the answer
        mov     bl, al
        cmp     al, 10
        je      check_answer
drain:                              # discard the rest of the line
        call    read_byte
        cmp     al, 10
        jne     drain
check_answer:
        cmp     bl, 'y'
        je      play_game
        cmp     bl, 'Y'
        je      play_game

quit:
        PRINT   msg_bye
        mov     eax, 60             # sys_exit
        xor     edi, edi
        syscall

# read_byte: read one byte from stdin into al; exits cleanly on EOF.
read_byte:
        xor     eax, eax            # sys_read
        xor     edi, edi            # stdin
        lea     rsi, [rip + inbuf]
        mov     edx, 1
        syscall
        test    rax, rax
        jle     quit
        movzx   eax, byte ptr [rip + inbuf]
        ret

# wait_enter: consume input up to and including a newline.
wait_enter:
        call    read_byte
        cmp     al, 10
        jne     wait_enter
        ret

# roll_die: xorshift64, returns a die value 1..6 in eax.
roll_die:
        mov     rax, [rip + seed]
        mov     rdx, rax
        shl     rdx, 13
        xor     rax, rdx
        mov     rdx, rax
        shr     rdx, 7
        xor     rax, rdx
        mov     rdx, rax
        shl     rdx, 17
        xor     rax, rdx
        mov     [rip + seed], rax
        shr     rax, 32             # use the high 32 bits
        xor     edx, edx
        mov     ecx, 6
        div     ecx                 # edx = eax mod 6
        lea     eax, [rdx + 1]
        ret

# print_int: print the unsigned integer in eax as decimal.
print_int:
        lea     rsi, [rip + numbuf + 24]    # rsi = end of buffer
        mov     rdi, rsi
        mov     ecx, 10
1:      xor     edx, edx
        div     ecx
        add     dl, '0'
        dec     rdi
        mov     [rdi], dl
        test    eax, eax
        jnz     1b
        mov     rdx, rsi
        sub     rdx, rdi            # length
        mov     rsi, rdi            # start of digits
        mov     eax, 1              # sys_write
        mov     edi, 1              # stdout
        syscall
        ret

        .section .note.GNU-stack,"",@progbits

### Assemble, then link

Assembly language is machine code written as readable words. Each line is one instruction for the processor. For example, `add eax, 1` means "add 1 to the number stored in `eax`". Names like `eax` are **registers**: tiny, very fast storage slots inside the CPU itself.

Two programs turn the text into something runnable:
1. The **assembler** (`as`) translates each line into its machine-code bytes and saves them in an *object file* (`.o`). Each line maps to exactly one instruction, so this is a direct, mechanical translation.
2. The **linker** (`ld`) packages the object file into an **executable**. It adds a header that tells the operating system where the program starts, and it fixes up the addresses of things in memory.

Notice what is missing: the C library. To print text, this program asks the operating system directly with a **system call**. It puts a code (`1` means "write") and some details into registers, then runs the `syscall` instruction.

In [ ]:
!as dice_game.s -o dice_game.o
!ld dice_game.o -o dice_asm

In [ ]:
!printf '\n\n\n\n\nn\n' | ./dice_asm

## 7. Machine code

Machine code is the real program: the bytes the CPU fetches and executes. Assembly is just a readable spelling of it. The file below is the whole dice game as **1,538 bytes of hex**, no assembler or linker involved.

How the bytes are laid out:

| Byte range | Contents |
|-----------|----------|
| 0-63 | **ELF header**, telling Linux this is a 64-bit x86 executable and where execution starts |
| 64-119 | **Program header**, telling the kernel to load the file at address `0x400000` and reserve extra zeroed memory for the variables |
| 120 onward | The program itself: instructions, then the text strings |

The first instruction is the two bytes `0f 31`, which is `rdtsc` (read the cycle counter), the very first line of our assembly. The next, `48 c1 e2 20`, is `shl rdx, 32`.

**What we get:** the `xxd -r -p` command turns the hex text back into raw bytes. Making the file executable and running it executes those bytes directly.

### What is hex, and how does a file become a program?

Every file is just a list of **bytes**, and each byte is a number from 0 to 255. People write bytes in **hexadecimal** (base 16), where two characters stand for one byte: `00` is 0, `0f` is 15, and `ff` is 255. The cell below holds the program as that list, written out as text, because you can't type raw bytes into a text file.

The cell after it uses `xxd -r -p` (the `r` means "reverse") to turn the hex text back into real bytes, then marks the file as runnable. The operating system doesn't care how a file was made. It looks at the first bytes (the **ELF header**), sees that they describe a valid program, copies the rest into memory, and starts running at the address the header names. That is how every program on Linux starts.

In [ ]:
%%writefile dice_game.hex
7f454c4602010100000000000000000002003e00010000007800400000000000
4000000000000000000000000000000000000000400038000100000000000000
0100000007000000000000000000000000004000000000000000400000000000
0206000000000000400600000000000000100000000000000f3148c1e2204809
d04883c801488905760500004531ed4531f6b801000000bf01000000488d35f8
030000ba240000000f05b801000000bf01000000488d3504040000ba26000000
0f0541bc01000000b801000000bf01000000488d350c040000ba070000000f05
4489e0e87e030000b801000000bf01000000488d3508050000ba010000000f05
b801000000bf01000000488d35db030000ba170000000f05e802030000e80703
00008905e2040000e8fc0200008905db040000e8f10200008905d4040000e8e6
0200008905cd0400008b05bb0400000305b90400008905bf0400008b05b10400
000305af0400008905b1040000b801000000bf01000000488d3585030000ba11
0000000f058b057f040000e8d6020000b801000000bf01000000488d355a0400
00ba030000000f058b0560040000e8b3020000b801000000bf01000000488d35
3a040000ba030000000f058b0549040000e890020000b801000000bf01000000
488d351a040000ba010000000f05b801000000bf01000000488d3515030000ba
110000000f058b0506040000e855020000b801000000bf01000000488d35d903
0000ba030000000f058b05e7030000e832020000b801000000bf01000000488d
35b9030000ba030000000f058b05cc030000e80f020000b801000000bf010000
00488d3599030000ba010000000f058b05a50300003b05a30300007f1c7c37b8
01000000bf01000000488d35c3020000ba120000000f05eb3841ffc5b8010000
00bf01000000488d3578020000ba140000000f05eb1b41ffc6b801000000bf01
000000488d356f020000ba1a0000000f05b801000000bf01000000488d358302
0000ba0b0000000f054489e8e875010000b801000000bf01000000488d356e02
0000ba0c0000000f054489f0e855010000b801000000bf01000000488d35df02
0000ba010000000f0541ffc44183fc050f8e92fdffffb801000000bf01000000
488d3535020000ba160000000f054539f57f1c7c34b801000000bf0100000048
8d355a020000ba140000000f05eb32b801000000bf01000000488d3512020000
ba120000000f05eb18b801000000bf01000000488d350a020000ba1c0000000f
05b801000000bf01000000488d3522020000ba130000000f05e84200000088c3
3c0a7409e8370000003c0a75f780fb790f84b6fcffff80fb590f84adfcffffb8
01000000bf01000000488d35f7010000ba140000000f05b83c00000031ff0f05
31c031ff488d3517020000ba010000000f054885c07ec80fb60504020000c3e8
dcffffff3c0a75f7c3488b05d20100004889c248c1e20d4831d04889c248c1ea
074831d04889c248c1e2114831d0488905ad01000048c1e82031d2b906000000
f7f18d4201c3488d35ce0100004889f7b90a00000031d2f7f180c23048ffcf88
1785c075f04889f24829fa4889feb801000000bf010000000f05c33d3d3d2044
6963652047616d653a20596f752076732e20436f6d7075746572203d3d3d0a42
657374206f66203520726f756e64733b206869676865737420746f74616c2077
696e732e0a0a526f756e6420507265737320456e74657220746f20726f6c6c2e
2e2e0a596f7520726f6c6c65643a202020202020436f6d707574657220726f6c
6c65643a20596f752077696e207468697320726f756e64210a436f6d70757465
722077696e73207468697320726f756e642e0a546965202d2d206e6f20706f69
6e74732e0a53636f72653a20596f7520202d20436f6d7075746572200a3d3d3d
2046696e616c20526573756c74203d3d3d0a596f752077696e20746865206761
6d65210a54686520636f6d70757465722077696e73207468652067616d652e0a
5468652067616d65206973206120647261772e0a0a506c617920616761696e3f
2028792f6e290a5468616e6b7320666f7220706c6179696e67210a202b20203d
200a

In [ ]:
!xxd -r -p dice_game.hex dice_mc
!chmod +x dice_mc
!ls -l dice_mc

In [ ]:
!printf '\n\n\n\n\nn\n' | ./dice_mc

### Peeking inside the machine code

The first cell shows the raw bytes of the program header and the start of the code. The second **disassembles** them: it translates the bytes back into assembly, so you can match each instruction to the bytes that encode it. Compare these with the first lines of `_start` in the assembly section above.

In [ ]:
!xxd dice_mc | head -12

In [ ]:
!objdump -D -b binary -mi386:x86-64 -M intel --start-address=0x78 --stop-address=0xb0 dice_mc | tail -n +8

# Part 2: Esoteric languages

The next four languages were designed as puzzles, jokes, or art rather than for getting work done. They all compute the same things as the languages above, but each one removes something you'd normally rely on.

Writing these by hand is impractical (try writing a number-printing routine in eight symbols), so each program here is **produced by a short Python generator** that you can read in the notebook. The generators work like macro assemblers: you write the game once in a readable form, and the generator emits the esoteric program.

**Running them:** one file, `esolang_runners.py`, holds a small interpreter for all four. I wrote the interpreters from the language specifications and checked every program against them. Interpreters differ in small details, so if you try the files in someone else's interpreter and something misbehaves, that is the first place to look.

### Words you'll need for this part

- **Esoteric** means deliberately strange. These languages aren't meant for real work. Each one is a puzzle that asks: how little do you need to be able to write any program?
- **Turing complete** means a language can compute anything any computer can, given enough time and memory. All four of these are, which is why they can play the game at all even though they look nothing like normal code.
- **Interpreter.** None of these languages has a ready-made compiler, so each program is run by an **interpreter**: a program that reads it and carries out each instruction. Ours is `esolang_runners.py`, written in Python. So the chain is: your esoteric program is read by a Python program, which is itself run by the Python interpreter, which runs on the CPU.
- **Stack machine.** Three of the four (Chef, Piet and Whitespace) work on a **stack**, the pile of values described earlier. `push` puts a number on top, and an instruction like `add` takes the top two numbers off, adds them, and pushes the result.
- **Generator.** A short Python program that writes the esoteric program for us, because writing them by hand would be painfully slow.

Run this cell to write the interpreters:

In [ ]:
%%writefile esolang_runners.py
"""esolang_runners.py: tiny interpreters for the dice-game programs.

    python esolang_runners.py bf   dice_game.bf     < input.txt
    python esolang_runners.py chef dice_game.chef
    python esolang_runners.py piet dice_game.png    < seed.txt      (Pillow + numpy; codel size auto-detected)
    python esolang_runners.py ws   dice_game.ws     < input.txt

Brainfuck input: one seed character, Enter, then Enter once per round (8-bit wrapping cells).
Chef input:      none (the recipe shuffles a bowl of 1-6 for each die).
Piet input:      one positive whole number (the seed).
Whitespace input: a positive seed number, Enter once per round, then n (no rematch).
"""
import sys

# ---------------- Brainfuck
def run_bf(src, inp, limit=200_000_000):
    code = [c for c in src if c in '+-<>[].,']
    jump, st = {}, []
    for i, c in enumerate(code):
        if c == '[': st.append(i)
        elif c == ']': j = st.pop(); jump[i] = j; jump[j] = i
    tape, p, pc, ip, out, steps = [0]*30000, 0, 0, 0, [], 0
    while pc < len(code):
        c = code[pc]
        if c == '+': tape[p] = (tape[p] + 1) & 255
        elif c == '-': tape[p] = (tape[p] - 1) & 255
        elif c == '>': p += 1
        elif c == '<': p -= 1
        elif c == '.': out.append(chr(tape[p]))
        elif c == ',':
            tape[p] = ord(inp[ip]) if ip < len(inp) else 0; ip += 1
        elif c == '[' and tape[p] == 0: pc = jump[pc]
        elif c == ']' and tape[p] != 0: pc = jump[pc]
        pc += 1; steps += 1
        if steps > limit: raise RuntimeError('step limit')
    return ''.join(out), steps

# ---------------- Chef
"""Small Chef interpreter (subset of the spec: everything the dice recipe uses)."""
import re, random

def parse_recipe(text):
    paras = [p.strip('\n') for p in re.split(r'\n\s*\n', text.strip('\n')) if p.strip()]
    ingredients, method, serves = {}, [], 0
    mode = None
    for p in paras:
        head = p.split('\n')[0].strip()
        if head == 'Ingredients.':
            for line in p.split('\n')[1:]:
                m = re.match(r'^(?:(-?\d+)\s+)?(?:(heaped|level)\s+)?(?:(g|kg|pinches|pinch|ml|l|dashes|dash|cups|cup|teaspoons|teaspoon|tablespoons|tablespoon)\s+)?(.+)$', line.strip())
                val, mtype, meas, name = m.groups()
                kind = 'dry' if meas in ('g', 'kg', 'pinch', 'pinches') or mtype else \
                       'liquid' if meas in ('ml', 'l', 'dash', 'dashes') else 'dry'
                ingredients[name] = [int(val) if val is not None else None, kind]
        elif head == 'Method.':
            body = ' '.join(p.split('\n')[1:])
            method = [x.strip() for x in body.split('.') if x.strip()]
        elif re.match(r'^Serves \d+\.$', head):
            serves = int(re.match(r'^Serves (\d+)', head).group(1))
    return ingredients, method, serves

def run_chef(text, stdin=None, rng=None):
    ing, method, serves = parse_recipe(text)
    rng = rng or random.Random()
    names = sorted(ing, key=len, reverse=True)
    N = '|'.join(re.escape(n) for n in names)
    bowls, dishes = {}, {}
    def bowl(n): return bowls.setdefault(int(n) if n else 1, [])
    def idx(g): return re.sub(r'\D', '', g) if g else None
    # loops: match starts with ends
    loops, stack = {}, []
    pat_start = re.compile(r'^(\w+) the (%s)$' % N)
    pat_end = re.compile(r'^(\w+)(?: the (%s))? until (\w+)$' % N)
    for i, st in enumerate(method):
        if pat_end.match(st) and ' until ' in st:
            j = stack.pop(); loops[i] = j; loops[j] = i
        elif pat_start.match(st) and st.split()[0] not in ('Put', 'Fold', 'Add', 'Remove', 'Combine', 'Divide', 'Liquefy', 'Clean', 'Mix', 'Stir', 'Pour', 'Take'):
            stack.append(i)
    pc = 0
    rx = lambda p: re.compile(r'^' + p + r'$')
    R = [
        ('put',   rx(r'Put (%s) into (?:the )?(?:(\d+)(?:st|nd|rd|th) )?mixing bowl' % N)),
        ('fold',  rx(r'Fold (%s) into (?:the )?(?:(\d+)(?:st|nd|rd|th) )?mixing bowl' % N)),
        ('add',   rx(r'Add (%s)(?: to (?:the )?(?:(\d+)(?:st|nd|rd|th) )?mixing bowl)?' % N)),
        ('remove',rx(r'Remove (%s)(?: from (?:the )?(?:(\d+)(?:st|nd|rd|th) )?mixing bowl)?' % N)),
        ('combine',rx(r'Combine (%s)(?: into (?:the )?(?:(\d+)(?:st|nd|rd|th) )?mixing bowl)?' % N)),
        ('divide',rx(r'Divide (%s)(?: into (?:the )?(?:(\d+)(?:st|nd|rd|th) )?mixing bowl)?' % N)),
        ('liq',   rx(r'Liquefy contents of the (?:(\d+)(?:st|nd|rd|th) )?mixing bowl')),
        ('mix',   rx(r'Mix (?:the )?(?:(\d+)(?:st|nd|rd|th) )?mixing bowl well')),
        ('clean', rx(r'Clean (?:the )?(?:(\d+)(?:st|nd|rd|th) )?mixing bowl')),
        ('pour',  rx(r'Pour contents of the (?:(\d+)(?:st|nd|rd|th) )?mixing bowl into the (?:(\d+)(?:st|nd|rd|th) )?baking dish')),
        ('aside', rx(r'Set aside')),
        ('fridge',rx(r'Refrigerate(?: for (\d+) hours?)?')),
    ]
    out = []
    def serve(n):
        for d in range(1, n + 1):
            for v, kind in reversed(dishes.get(d, [])):
                out.append(chr(v) if kind == 'liquid' else str(v))
    def val(name):
        v = ing[name][0]
        if v is None: raise RuntimeError('undefined ingredient ' + name)
        return v
    while pc < len(method):
        st = method[pc]
        if pc in loops and loops[pc] > pc:                      # loop start
            m = pat_start.match(st)
            if val(m.group(2)) == 0: pc = loops[pc] + 1; continue
            pc += 1; continue
        if pc in loops and loops[pc] < pc:                      # loop end
            m = pat_end.match(st)
            if m.group(2): ing[m.group(2)][0] -= 1
            start = loops[pc]
            if val(pat_start.match(method[start]).group(2)) != 0: pc = start + 1
            else: pc += 1
            continue
        for kind, r in R:
            m = r.match(st)
            if m: break
        else:
            raise RuntimeError('cannot parse: ' + st)
        g = m.groups()
        if kind == 'put': bowl(g[1]).append((val(g[0]), ing[g[0]][1]))
        elif kind == 'fold': ing[g[0]][0] = bowl(g[1]).pop()[0]
        elif kind in ('add', 'remove', 'combine', 'divide'):
            b = bowl(g[1]); v, k = b[-1]; x = val(g[0])
            b[-1] = ({'add': v + x, 'remove': v - x, 'combine': v * x, 'divide': v // x if x else 0}[kind], k)
        elif kind == 'liq': bowls[int(g[0]) if g[0] else 1] = [(v, 'liquid') for v, _ in bowl(g[0])]
        elif kind == 'mix': rng.shuffle(bowl(g[0]))
        elif kind == 'clean': bowls[int(g[0]) if g[0] else 1] = []
        elif kind == 'pour':
            dishes.setdefault(int(g[1]) if g[1] else 1, []).extend(bowl(g[0]))
        elif kind == 'aside':
            depth, j = 0, pc
            # jump past the matching 'until' of the innermost loop containing pc
            starts = [s for s in loops if loops[s] > s and s < pc < loops[s]]
            pc = loops[max(starts)] + 1; continue
        elif kind == 'fridge':
            if g[0]: serve(int(g[0]))
            break
        pc += 1
    else:
        serve(serves)
    return ''.join(out)


# ---------------- Piet
"""Small Piet interpreter.  Usage: python pietrun.py dice_game.png [codel_size]  (codel size is auto-detected)"""
import math
import numpy as np
from PIL import Image

HUE_LIGHT = {}
_base = [(255,192,192),(255,0,0),(192,0,0),(255,255,192),(255,255,0),(192,192,0),
         (192,255,192),(0,255,0),(0,192,0),(192,255,255),(0,255,255),(0,192,192),
         (192,192,255),(0,0,255),(0,0,192),(255,192,255),(255,0,255),(192,0,192)]
for i, rgb in enumerate(_base): HUE_LIGHT[rgb] = (i // 3, i % 3)
WHITE, BLACK = (255,255,255), (0,0,0)
CMDS = [[None,'push','pop'],['add','sub','mul'],['div','mod','not'],['greater','pointer','switch'],
        ['dup','roll','innum'],['inchar','outnum','outchar']]
DIRS = [(1,0),(0,1),(-1,0),(0,-1)]       # DP: right, down, left, up (clockwise)

def detect_codel(a):
    g = 0
    for arr in (a, a.transpose(1, 0, 2)):
        diff = np.any(arr[:, 1:] != arr[:, :-1], axis=2)
        for row in diff:
            edges = np.flatnonzero(row) + 1
            if len(edges):
                bounds = np.concatenate(([0], edges, [arr.shape[1]]))
                for r in np.diff(bounds): g = math.gcd(g, int(r))
        if g == 1: break
    return max(g, 1)

def load_piet(path, codel=None):
    a = np.array(Image.open(path).convert('RGB'))
    c = codel or detect_codel(a)
    a = a[::c, ::c]
    h, w = a.shape[:2]
    grid = [[tuple(int(v) for v in a[y, x]) for x in range(w)] for y in range(h)]
    return grid, w, h

def run_piet(grid, w, h, stdin=sys.stdin, stdout=sys.stdout, limit=5_000_000):
    def kind(x, y):
        if not (0 <= x < w and 0 <= y < h): return 'black'
        c = grid[y][x]
        if c == BLACK: return 'black'
        if c == WHITE or c not in HUE_LIGHT: return 'white'
        return 'colour'
    block_of, blocks = {}, []
    def block(x, y):
        if (x, y) in block_of: return blocks[block_of[(x, y)]]
        col, todo, cells = grid[y][x], [(x, y)], set()
        while todo:
            p = todo.pop()
            if p in cells: continue
            cells.add(p)
            for dx, dy in DIRS:
                q = (p[0] + dx, p[1] + dy)
                if 0 <= q[0] < w and 0 <= q[1] < h and q not in cells and grid[q[1]][q[0]] == col: todo.append(q)
        for p in cells: block_of[p] = len(blocks)
        blocks.append(cells); return cells
    def exit_codel(cells, dp, cc):
        dx, dy = DIRS[dp]
        key = lambda p: p[0] * dx + p[1] * dy                       # furthest in DP direction
        best = max(key(p) for p in cells)
        edge = [p for p in cells if key(p) == best]
        cdx, cdy = DIRS[(dp + (3 if cc == 0 else 1)) % 4]            # cc 0 = left (counterclockwise), 1 = right
        return max(edge, key=lambda p: p[0] * cdx + p[1] * cdy)
    stack, dp, cc, pos, steps = [], 0, 0, (0, 0), 0
    def tokens():
        while True:
            line = stdin.readline()
            if not line: return
            for t in line.split(): yield t
    toks = tokens()
    while steps < limit:
        steps += 1
        cells = block(*pos)
        moved = False
        for attempt in range(8):
            ex = exit_codel(cells, dp, cc)
            nx, ny = ex[0] + DIRS[dp][0], ex[1] + DIRS[dp][1]
            k = kind(nx, ny)
            if k == 'colour':
                a, b = HUE_LIGHT[grid[pos[1]][pos[0]]], HUE_LIGHT[grid[ny][nx]]
                cmd = CMDS[(b[0] - a[0]) % 6][(b[1] - a[1]) % 3]
                size = len(cells)
                pos, moved = (nx, ny), True
                execute(cmd, size, stack, stdout, toks, lambda: None)
                if cmd == 'pointer': pass
                break
            if k == 'white':
                while kind(nx, ny) == 'white': nx += DIRS[dp][0]; ny += DIRS[dp][1]
                if kind(nx, ny) == 'colour': pos, moved = (nx, ny), True; break
            if attempt % 2 == 0: cc ^= 1
            else: dp = (dp + 1) % 4
        if not moved: return
        # state changes from pointer/switch are applied inside execute via the shared dict
        dp, cc = STATE['dp_fn'](dp, cc)
    raise RuntimeError('step limit')

STATE = {'dp_fn': lambda dp, cc: (dp, cc)}
_pending = {'dp': 0, 'cc': 0}

def execute(cmd, size, st, out, toks, _):
    _pending['dp'] = 0; _pending['cc'] = 0
    if cmd == 'push': st.append(size)
    elif cmd == 'pop':
        if st: st.pop()
    elif cmd in ('add', 'sub', 'mul', 'div', 'mod', 'greater'):
        if len(st) >= 2:
            b, a = st.pop(), st.pop()
            if cmd in ('div', 'mod') and b == 0: st.extend([a, b]); return
            st.append({'add': a + b, 'sub': a - b, 'mul': a * b, 'greater': int(a > b)}.get(cmd, None)
                      if cmd in ('add', 'sub', 'mul', 'greater') else (a // b if cmd == 'div' else a % b))
    elif cmd == 'not':
        if st: st.append(int(st.pop() == 0))
    elif cmd == 'pointer':
        if st: _pending['dp'] = st.pop() % 4
    elif cmd == 'switch':
        if st: _pending['cc'] = abs(st.pop()) % 2
    elif cmd == 'dup':
        if st: st.append(st[-1])
    elif cmd == 'roll':
        if len(st) >= 2:
            n, d = st.pop(), st.pop()
            if d < 0 or d > len(st): st.extend([d, n]); return
            if d > 0:
                n %= d
                for _ in range(n):
                    v = st.pop()
                    st.insert(len(st) - (d - 1), v)
    elif cmd == 'innum':
        t = next(toks, None)
        if t is not None: st.append(int(t))
    elif cmd == 'inchar': pass
    elif cmd == 'outnum':
        if st: out.write(str(st.pop())); out.flush()
    elif cmd == 'outchar':
        if st: out.write(chr(st.pop())); out.flush()
STATE['dp_fn'] = lambda dp, cc: ((dp + _pending['dp']) % 4, cc ^ _pending['cc'])


# ---------------- Whitespace

def parse_ws(src):
    toks = [c for c in src if c in ' \t\n']
    i, prog = 0, []
    def num():
        nonlocal i
        sign = -1 if toks[i] == '\t' else 1; i += 1
        v = 0
        while toks[i] != '\n':
            v = v * 2 + (toks[i] == '\t'); i += 1
        i += 1
        return sign * v
    def lab():
        nonlocal i
        s = ''
        while toks[i] != '\n':
            s += toks[i]; i += 1
        i += 1
        return s
    def nxt(n):
        nonlocal i
        s = ''.join(toks[i:i+n]); i += n
        return s
    while i < len(toks):
        a = nxt(1)
        if a == ' ':
            b = nxt(1)
            if b == ' ': prog.append(('push', num()))
            else:
                prog.append({' ': 'dup', '\t': 'swap', '\n': 'drop'}[nxt(1)])
        elif a == '\t':
            b = nxt(1)
            if b == ' ':
                prog.append({'  ': 'add', ' \t': 'sub', ' \n': 'mul',
                             '\t ': 'div', '\t\t': 'mod'}[nxt(2)])
            elif b == '\t':
                prog.append({' ': 'store', '\t': 'load'}[nxt(1)])
            else:
                prog.append({'  ': 'outc', ' \t': 'outn',
                             '\t ': 'readc', '\t\t': 'readn'}[nxt(2)])
        else:
            b = nxt(2)
            kind = {'  ': 'label', ' \t': 'call', ' \n': 'jump', '\t ': 'jz',
                    '\t\t': 'jneg', '\t\n': 'ret', '\n\n': 'end'}[b]
            prog.append((kind, lab()) if kind in ('label','call','jump','jz','jneg') else kind)
    return prog

def run_ws(prog, stdin=sys.stdin, stdout=sys.stdout):
    labels = {p[1]: n for n, p in enumerate(prog) if isinstance(p, tuple) and p[0] == 'label'}
    st, calls, heap, pc = [], [], {}, 0
    while pc < len(prog):
        p = prog[pc]; pc += 1
        k, arg = (p if isinstance(p, tuple) else (p, None))
        if k == 'push': st.append(arg)
        elif k == 'dup': st.append(st[-1])
        elif k == 'swap': st[-1], st[-2] = st[-2], st[-1]
        elif k == 'drop': st.pop()
        elif k in ('add','sub','mul','div','mod'):
            b, a = st.pop(), st.pop()
            st.append({'add': a+b, 'sub': a-b, 'mul': a*b}.get(k) if k in ('add','sub','mul')
                      else (a // b if k == 'div' else a % b))
        elif k == 'store': v, a = st.pop(), st.pop(); heap[a] = v
        elif k == 'load': st.append(heap.get(st.pop(), 0))
        elif k == 'call': calls.append(pc); pc = labels[arg]
        elif k == 'jump': pc = labels[arg]
        elif k == 'jz':
            if st.pop() == 0: pc = labels[arg]
        elif k == 'jneg':
            if st.pop() < 0: pc = labels[arg]
        elif k == 'ret': pc = calls.pop()
        elif k == 'end': break
        elif k == 'outc': stdout.write(chr(st.pop())); stdout.flush()
        elif k == 'outn': stdout.write(str(st.pop())); stdout.flush()
        elif k == 'readc':
            ch = stdin.read(1); heap[st.pop()] = ord(ch) if ch else -1
        elif k == 'readn':
            heap[st.pop()] = int(stdin.readline().strip() or 0)


if __name__ == '__main__':
    kind, path = sys.argv[1], sys.argv[2]
    if kind == 'bf':
        print(run_bf(open(path).read(), sys.stdin.read())[0], end='')
    elif kind == 'chef':
        print(run_chef(open(path).read()), end='')
    elif kind == 'piet':
        g, w, h = load_piet(path)
        run_piet(g, w, h)
    elif kind == 'ws':
        run_ws(parse_ws(open(path).read()))
    else:
        sys.exit('first argument must be bf, chef, piet or ws')

## 8. Brainfuck

Brainfuck (Urban Müller, 1993) has **eight commands** and nothing else:

| Command | Meaning |
|---------|---------|
| `>` `<` | Move the pointer one cell right or left along a tape of byte cells |
| `+` `-` | Add or subtract 1 in the current cell (wrapping at 256) |
| `.` | Print the current cell as a character |
| `,` | Read one character into the current cell |
| `[` `]` | Loop: repeat while the current cell isn't zero |

There are no variables, no numbers except what you count out with `+`, no strings, no `if`, and no random numbers.

How the game works anyway:
- **Variables** are tape positions. The generator hands out cells by name and works out the `>`/`<` moves.
- **`if` and `else`** are built from loops that zero a flag cell on exit.
- **Printing text** adjusts one scratch cell by the difference between consecutive characters, then prints it.
- **Random numbers** come from `seed = seed * 101 + 13`, which relies on the cell wrapping at 256. A die is `seed / 43 + 1`, found by counting up to 43 and rolling over, since there is no divide.
- **Comparing totals** counts both down together until one hits zero.

**Input:** one seed character and Enter, then Enter once per round. If you give it too little input, it waits forever, as any Brainfuck program would.

**What we get:** a 26,630-command text file that plays five rounds and announces a winner. It has no rematch loop.

### A tiny Brainfuck program first

Before the full game, here is a one-line program so you can see the idea. It prints the letter `A`, whose character code is 65.

```
++++++++[>++++++++<-]>+.
```

- `++++++++` adds 1 eight times, so cell 0 holds 8.
- `[` starts a loop that repeats while the current cell isn't zero. Inside it, `>` moves to cell 1, `++++++++` adds 8 there, `<` moves back, and `-` subtracts 1 from cell 0. The loop runs 8 times, so cell 1 ends up at 8 x 8 = 64.
- `>` moves to cell 1, `+` makes it 65, and `.` prints the character with code 65, which is `A`.

In [ ]:
!echo '++++++++[>++++++++<-]>+.' > tiny.bf
!python3 esolang_runners.py bf tiny.bf

In [ ]:
%%writefile bf_gen.py
# Brainfuck dice game generator: a tiny "macro assembler" that emits plain BF.
class BF:
    def __init__(self):
        self.out = []
        self.ptr = 0
        self.cells = {}
    def cell(self, name):
        if name not in self.cells:
            self.cells[name] = len(self.cells)
        return self.cells[name]
    # ---- primitives
    def goto(self, c):
        c = self.cell(c) if isinstance(c, str) else c
        d = c - self.ptr
        self.out.append(('>' if d > 0 else '<') * abs(d))
        self.ptr = c
    def emit(self, s): self.out.append(s)
    def clear(self, c): self.goto(c); self.emit('[-]')
    def inc(self, c, n=1): self.goto(c); self.emit('+' * n)
    def dec(self, c, n=1): self.goto(c); self.emit('-' * n)
    def setv(self, c, n): self.clear(c); self.inc(c, n)
    def put(self, c): self.goto(c); self.emit('.')
    def get(self, c): self.goto(c); self.emit(',')
    def move(self, src, dst):          # dst += src; src = 0
        self.goto(src); self.emit('[-'); self.goto(dst); self.emit('+'); self.goto(src); self.emit(']')
    def copy(self, src, dst, tmp):     # dst = src (src preserved)
        self.clear(dst)
        self.goto(src); self.emit('[-'); self.goto(dst); self.emit('+'); self.goto(tmp); self.emit('+'); self.goto(src); self.emit(']')
        self.move(tmp, src)
    def add_copy(self, src, dst, tmp): # dst += src (src preserved)
        self.goto(src); self.emit('[-'); self.goto(dst); self.emit('+'); self.goto(tmp); self.emit('+'); self.goto(src); self.emit(']')
        self.move(tmp, src)
    # ---- control flow (every construct ends with the pointer on a known cell)
    def while_(self, c, body):
        self.goto(c); self.emit('['); body(); self.goto(c); self.emit(']')
    def if_(self, c, body):            # consumes c
        self.goto(c); self.emit('['); body(); self.clear(c); self.emit(']')
    def if_else(self, c, tmp, then, els):   # consumes c
        self.setv(tmp, 1)
        self.goto(c); self.emit('['); then(); self.dec(tmp); self.clear(c); self.emit(']')
        self.goto(tmp); self.emit('['); els(); self.dec(tmp); self.emit(']')
    # ---- text
    def say(self, s):
        self.clear('P')
        cur = 0
        for ch in s:
            d = (ord(ch) - cur) % 256
            self.goto('P')
            self.emit('+' * d if d <= 128 else '-' * (256 - d))
            self.emit('.')
            cur = ord(ch)
    def digits(self, c):               # print a cell holding 0..99
        t, f, s1, s2, tm = 'd_t', 'd_flag', 'd_s', 'd_s2', 'd_tmp'
        self.copy(c, t, 'd_c')
        # f = 1 if t >= 10
        self.setv(f, 1)
        self.copy(t, 'd_w', 'd_c')
        for _ in range(10):
            self.copy('d_w', s1, 'd_c')
            self.if_else(s1, tm, lambda: self.dec('d_w'), lambda: self.clear(f))
        self.if_(f, lambda: (self.setv('d_o', 49), self.put('d_o'), self.clear('d_o'), self.dec(t, 10)))
        self.setv('d_o', 48); self.add_copy(t, 'd_o', 'd_c'); self.put('d_o'); self.clear('d_o')
    def wait_enter(self):
        self.get('D'); self.dec('D', 10)
        self.while_('D', lambda: (self.get('D'), self.dec('D', 10)))
    def roll(self, dst):
        # seed = seed*101 + 13 (mod 256, relies on 8-bit wraparound)
        self.clear('r_t'); self.move('seed', 'r_t')
        self.goto('r_t'); self.emit('['); self.dec('r_t'); self.inc('seed', 101); self.goto('r_t'); self.emit(']')
        self.inc('seed', 13)
        # q = seed // 43  (count up to 43, then roll over)
        self.copy('seed', 'r_n', 'r_c'); self.clear('r_r'); self.clear('r_q')
        def body():
            self.dec('r_n'); self.inc('r_r')
            self.copy('r_r', 'r_s', 'r_c'); self.dec('r_s', 43)
            self.if_else('r_s', 'r_tm', lambda: None, lambda: (self.clear('r_r'), self.inc('r_q')))
        self.while_('r_n', body)
        self.clear(dst); self.move('r_q', dst); self.inc(dst)
    def compare(self, a, b, on_a, on_b, on_tie):
        self.copy(a, 'x', 'c_c'); self.copy(b, 'y', 'c_c')
        for _ in range(12):
            self.copy('x', 'c_s1', 'c_c'); self.copy('y', 'c_s2', 'c_c'); self.clear('c_f')
            self.if_('c_s1', lambda: self.if_('c_s2', lambda: self.setv('c_f', 1)))
            self.if_('c_f', lambda: (self.dec('x'), self.dec('y')))
        self.copy('x', 'c_s1', 'c_c')
        def els():
            self.copy('y', 'c_s2', 'c_c')
            self.if_else('c_s2', 'c_tb', on_b, on_tie)
        self.if_else('c_s1', 'c_ta', on_a, els)

def build(rounds=5):
    b = BF()
    for n in ['P', 'D']: b.cell(n)
    b.say("Type any character as the seed, then press Enter: ")
    b.get('seed'); b.dec('D', 0)
    b.get('D'); b.dec('D', 10)
    b.while_('D', lambda: (b.get('D'), b.dec('D', 10)))
    b.say("=== Dice Game: You vs. Computer ===\n")
    b.say("Best of %d rounds; highest total wins.\n" % rounds)
    b.setv('left', rounds)
    def one_round():
        b.inc('rn')
        b.say("\nRound "); b.digits('rn'); b.say("\n")
        b.say("Press Enter to roll...\n"); b.wait_enter()
        for v in ('p1', 'p2', 'c1', 'c2'): b.roll(v)
        b.copy('p1', 'pt', 'm_c'); b.add_copy('p2', 'pt', 'm_c')
        b.copy('c1', 'ct', 'm_c'); b.add_copy('c2', 'ct', 'm_c')
        b.say("You rolled:      "); b.digits('p1'); b.say(" + "); b.digits('p2'); b.say(" = "); b.digits('pt'); b.say("\n")
        b.say("Computer rolled: "); b.digits('c1'); b.say(" + "); b.digits('c2'); b.say(" = "); b.digits('ct'); b.say("\n")
        b.compare('pt', 'ct',
                  lambda: (b.say("You win this round!\n"), b.inc('pw')),
                  lambda: (b.say("Computer wins this round.\n"), b.inc('cw')),
                  lambda: b.say("Tie -- no points.\n"))
        b.say("Score: You "); b.digits('pw'); b.say(" - Computer "); b.digits('cw'); b.say("\n")
        b.dec('left')
    b.while_('left', one_round)
    b.say("\n=== Final Result ===\n")
    b.compare('pw', 'cw',
              lambda: b.say("You win the game!\n"),
              lambda: b.say("The computer wins the game.\n"),
              lambda: b.say("The game is a draw.\n"))
    b.say("Thanks for playing!\n")
    return ''.join(b.out), b.cells

if __name__ == '__main__':
    code, cells = build()
    # wrap for readability
    header = ("[ Dice game: you vs. the computer, in Brainfuck. 5 rounds, highest two-dice total wins.\n"
              "  Input: one character (the seed) + Enter, then Enter once per round.\n"
              "  Needs 8-bit wrapping cells (the usual default). Only the 8 BF symbols are code;\n"
              "  everything in this comment is ignored, so it avoids those symbols. ]\n")
    header = header.replace('.', ' ').replace(',', ' ').replace('+', 'plus').replace('-', ' ').replace('[', '(').replace(']', ')')
    lines = [code[i:i+72] for i in range(0, len(code), 72)]
    open('dice_game.bf', 'w').write(header + '\n'.join(lines) + '\n')
    print(len(code), 'BF commands;', len(cells), 'cells used')

In [ ]:
!python3 bf_gen.py
!head -c 560 dice_game.bf

In [ ]:
!printf 'A\n\n\n\n\n\n' | python3 esolang_runners.py bf dice_game.bf

Change the `A` to any other character and you get a different game.

## 9. Chef

Chef (David Morgan-Mar, 2002) makes programs look like **cooking recipes**. The parts of a recipe are the parts of the machine:

| In the recipe | In the machine |
|---------------|----------------|
| Ingredients | Variables. *Dry* ingredients print as numbers, *liquid* ones as characters |
| Mixing bowls | Stacks (`Put`, `Fold`, `Add`, `Combine`, `Remove`) |
| Baking dishes | Output stacks, printed by `Serves` |
| `Verb the x. ... Verb until verbed.` | A loop that runs while `x` is not zero |

There is **no `if`**. Chef has only loops, so a conditional is a loop that runs at most once and leaves with `Set aside`.

How the game works:
- **Dice:** Chef has one built-in source of randomness, `Mix the mixing bowl well`. We put the numbers 1 to 6 in a bowl, shuffle, and take the top one. No generator or seed is needed.
- **Text:** every character the game prints is a liquid ingredient with its code (for example `89 ml upper y`). A bowl prints from the top down, so each line is pushed backwards.
- **Comparing totals:** repeatedly subtract 1 from both totals until one reaches zero. Whichever isn't zero is bigger.
- **Output order:** each round goes into its own baking dish, and `Serves 7` prints the seven dishes in order at the end. So the whole game appears when the recipe finishes.

**Input:** none.

**What we get:** a recipe about 52 KB long. It plays five rounds and announces a winner, with no rematch.

In [ ]:
%%writefile chef_gen.py
# Generates the Chef recipe for the dice game.
ROUNDS = 5

NUM = {1: 'one', 2: 'two', 3: 'three', 4: 'four', 5: 'five', 6: 'six'}
DIGIT_WORDS = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine']
PUNCT = {' ': 'space', '\n': 'newline', '=': 'equals', '.': 'point', ':': 'colon',
         ';': 'semicolon', '!': 'bang', '+': 'plus', '-': 'hyphen'}

def charname(ch):
    if ch.isupper(): return 'upper ' + ch.lower()
    if ch.islower(): return 'lower ' + ch
    if ch.isdigit(): return 'digit ' + DIGIT_WORDS[int(ch)]
    return 'mark ' + PUNCT[ch]

used_chars = []
def need(ch):
    if ch not in used_chars: used_chars.append(ch)
    return charname(ch)

VARS = ['your first die', 'your second die', 'computer first die', 'computer second die',
        'your total', 'computer total', 'left count', 'right count', 'product',
        'your wins', 'computer wins', 'player flag', 'computer flag', 'tie flag']

steps = []
def s(text): steps.append(text)
BOWL = 'the mixing bowl'

def ordinal(n):
    suf = 'th' if 10 <= n % 100 <= 20 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')
    return '%d%s' % (n, suf)

def copy(src, dst):
    s('Put %s into %s.' % (src, BOWL)); s('Fold %s into %s.' % (dst, BOWL))

def roll(var):
    s('Clean %s.' % BOWL)
    for n in range(6, 0, -1): s('Put %s into %s.' % (NUM[n], BOWL))
    s('Mix %s well.' % BOWL)
    s('Fold %s into %s.' % (var, BOWL))

def push_line(items):
    """Items are printed in the order given; the bowl prints top-first, so push them backwards."""
    for kind, val in reversed(items):
        if kind == 's':
            for ch in reversed(val): s('Put %s into %s.' % (need(ch), BOWL))
        else:
            s('Put %s into %s.' % (val, BOWL))

def if_flag(flag, verb, body):
    s('%s the %s.' % (verb, flag)); body(); s('Set aside.'); s('%s until %sed.' % (verb, verb.lower()))

def increment(var):
    s('Put %s into %s.' % (var, BOWL)); s('Add one to %s.' % BOWL); s('Fold %s into %s.' % (var, BOWL))

def compare(a, b, score):
    """Sets player flag / computer flag / tie flag from the two values. Optionally counts a win."""
    copy(a, 'left count'); copy(b, 'right count')
    copy('zero', 'player flag'); copy('zero', 'computer flag'); copy('one', 'tie flag')
    s('Put left count into %s.' % BOWL); s('Combine right count into %s.' % BOWL); s('Fold product into %s.' % BOWL)
    s('Simmer the product.')
    for v in ('left count', 'right count'):
        s('Put %s into %s.' % (v, BOWL)); s('Remove one from %s.' % BOWL); s('Fold %s into %s.' % (v, BOWL))
    s('Put left count into %s.' % BOWL); s('Combine right count into %s.' % BOWL); s('Fold product into %s.' % BOWL)
    s('Simmer until simmered.')
    for side, flag, wins in (('left count', 'player flag', 'your wins'), ('right count', 'computer flag', 'computer wins')):
        s('Roast the %s.' % side)
        if score: increment(wins)
        copy('one', flag); copy('zero', 'tie flag')
        s('Set aside.'); s('Roast until roasted.')

def outcome(texts):
    for flag, verb, text in zip(('player flag', 'computer flag', 'tie flag'), ('Grill', 'Toast', 'Poach'), texts):
        if_flag(flag, verb, lambda t=text: push_line([('s', t)]))

dish = 0
def pour():
    global dish
    dish += 1
    s('Pour contents of %s into the %s baking dish.' % (BOWL, ordinal(dish))); s('Clean %s.' % BOWL)

# ---- header (1st dish)
s('Clean %s.' % BOWL)
push_line([('s', "=== Dice Game: You vs. Computer ===\n"), ('s', "Best of %d rounds; highest total wins.\n" % ROUNDS)])
pour()

# ---- rounds (one dish each, unrolled)
for k in range(1, ROUNDS + 1):
    for v in ('your first die', 'your second die', 'computer first die', 'computer second die'): roll(v)
    s('Clean %s.' % BOWL)
    s('Put your first die into %s.' % BOWL); s('Add your second die to %s.' % BOWL); s('Fold your total into %s.' % BOWL)
    s('Put computer first die into %s.' % BOWL); s('Add computer second die to %s.' % BOWL); s('Fold computer total into %s.' % BOWL)
    compare('your total', 'computer total', score=True)
    s('Clean %s.' % BOWL)
    push_line([('s', 'Score: You '), ('v', 'your wins'), ('s', ' - Computer '), ('v', 'computer wins'), ('s', '\n')])
    outcome(["You win this round!\n", "Computer wins this round.\n", "Tie -- no points.\n"])
    push_line([('s', 'Computer rolled: '), ('v', 'computer first die'), ('s', ' + '), ('v', 'computer second die'),
               ('s', ' = '), ('v', 'computer total'), ('s', '\n')])
    push_line([('s', 'You rolled:      '), ('v', 'your first die'), ('s', ' + '), ('v', 'your second die'),
               ('s', ' = '), ('v', 'your total'), ('s', '\n')])
    push_line([('s', '\nRound '), ('v', NUM[k]), ('s', '\n')])
    pour()

# ---- final result (last dish)
compare('your wins', 'computer wins', score=False)
s('Clean %s.' % BOWL)
push_line([('s', 'Thanks for playing!\n')])
outcome(["You win the game!\n", "The computer wins the game.\n", "The game is a draw.\n"])
push_line([('s', '\n=== Final Result ===\n')])
pour()

# ---- assemble recipe
ingredients = ['%d g %s' % (n, NUM[n]) for n in range(1, 7)] + ['0 g zero'] + ['0 g %s' % v for v in VARS]
ingredients += ['%d ml %s' % (ord(ch), charname(ch)) for ch in used_chars]

recipe = []
recipe.append("Dice Game Casserole.\n")
recipe.append("You and the computer each roll two dice per round. Dice are rolled by shuffling a bowl\n"
              "of the numbers one to six. The higher total wins the round. Results are plated into\n"
              "one baking dish per round and served together at the end, so the whole game prints\n"
              "when the recipe finishes. No input is needed.\n")
recipe.append("Ingredients.\n" + '\n'.join(ingredients) + "\n")
recipe.append("Method.\n" + '\n'.join(steps) + "\n")
recipe.append("Serves %d.\n" % dish)
open('dice_game.chef', 'w').write('\n'.join(recipe))
print(len(steps), 'steps;', len(ingredients), 'ingredients;', dish, 'dishes;', len('\n'.join(recipe)), 'bytes')

In [ ]:
!python3 chef_gen.py
!head -16 dice_game.chef
!sed -n '/^Method/,+10p' dice_game.chef

In [ ]:
!python3 esolang_runners.py chef dice_game.chef

Run it again and the rolls change, because the bowl is shuffled fresh each time.

## 10. Piet

Piet (David Morgan-Mar, 2001) programs are **images**, named after the painter Piet Mondrian. There is no text at all.

How it works:
- The image is made of colored **blocks** of same-colored squares (codels). Execution walks from block to block.
- Which instruction runs is decided by **how the color changes** between two blocks: the hue change picks a row (add, divide, print, ...) and the lightness change picks a column.
- It's a stack machine. `push` pushes the **size of the block you just left**, so to push 72 you must draw a block of 72 squares.
- A *direction pointer* says which way to travel. The `pointer` instruction turns it, which is how Piet branches.
- Black squares are walls. The program ends when the pointer is stuck.

How the game works here:
- The main path runs along the top, with tall blocks for big numbers (letters are ASCII codes like 72 or 111).
- At the end, `greater` then `pointer` turns the path **down**. The "You win!" and "Computer wins!" messages hang below as long columns, and the tie message continues along the top.
- Dice use `seed = (seed + 1) * 75 mod 65537 - 1`, with a few warm-up steps so small seeds don't start out low.
- Each ending is a three-square block entered in the middle, so the pointer can't back out and the program stops.

**Input:** one positive whole number (the seed).

**What we get:** an image of about 968 x 379 codels. It plays **one round** (two dice each), then announces who won.

> The codel size is 6 pixels. If you load the image in another Piet interpreter, set that value if it asks.

In [ ]:
%%writefile piet_gen.py
# Piet dice game: builds the program as a list of stack operations, then lays them out as coloured blocks.
from PIL import Image
import sys

HUES = ['red', 'yellow', 'green', 'cyan', 'blue', 'magenta']
RGB = {  # (hue, lightness) lightness: 0 light, 1 normal, 2 dark
    (0,0):(255,192,192),(0,1):(255,0,0),(0,2):(192,0,0),
    (1,0):(255,255,192),(1,1):(255,255,0),(1,2):(192,192,0),
    (2,0):(192,255,192),(2,1):(0,255,0),(2,2):(0,192,0),
    (3,0):(192,255,255),(3,1):(0,255,255),(3,2):(0,192,192),
    (4,0):(192,192,255),(4,1):(0,0,255),(4,2):(0,0,192),
    (5,0):(255,192,255),(5,1):(255,0,255),(5,2):(192,0,192)}
CMD = {'push':(0,1),'pop':(0,2),'add':(1,0),'sub':(1,1),'mul':(1,2),'div':(2,0),'mod':(2,1),'not':(2,2),
       'greater':(3,0),'pointer':(3,1),'switch':(3,2),'dup':(4,0),'roll':(4,1),'innum':(4,2),
       'inchar':(5,0),'outnum':(5,1),'outchar':(5,2)}

# ---------- program as stack operations
def P(n): return [('push', n)]
def O(name): return [(name,)]
def say(text):
    ops = []
    for ch in text: ops += P(ord(ch)) + O('outchar')
    return ops
def const65536(): return P(16) + O('dup') + O('mul') + O('dup') + O('mul')
def swap(): return P(2) + P(1) + O('roll')
def pick(k): return P(k) + P(k - 1) + O('roll') + O('dup') + P(k + 1) + P(1) + O('roll')
def zero(): return P(1) + O('not')
def die_step():
    # stack [.., T, seed]  ->  [.., seed', T + die]   (prints the die)
    ops = swap()
    ops += P(1) + O('add') + P(75) + O('mul') + const65536() + P(1) + O('add') + O('mod') + P(1) + O('sub')
    ops += O('dup') + P(6) + O('mul') + const65536() + O('div') + P(1) + O('add')
    ops += O('dup') + O('outnum')
    ops += P(3) + P(2) + O('roll') + O('add')
    return ops

def seed_step():
    # seed = ((seed + 1) * 75 mod 65537) - 1
    return P(1) + O('add') + P(75) + O('mul') + const65536() + P(1) + O('add') + O('mod') + P(1) + O('sub')
main = O('innum') + const65536() + O('mod')
main += seed_step() + seed_step() + seed_step() + seed_step()      # warm-up: small seeds start out low
main += zero()
main += say('You rolled: ') + die_step() + say(' + ') + die_step() + say(' = ') + O('dup') + O('outnum') + say('\n')
main += swap() + zero()
main += say('Computer rolled: ') + die_step() + say(' + ') + die_step() + say(' = ') + O('dup') + O('outnum') + say('\n')
main += swap() + O('pop')
main += pick(2) + pick(2) + O('greater') + O('pointer')       # 1 -> DP turns down (you win)

NOP = P(1) + O('pop')                                          # first ops after a branch (leaving a 1-codel block)
down1  = NOP + say('You win!\n')
right1 = NOP + swap() + O('greater') + O('pointer')            # 1 -> DP turns down (computer wins)
down2  = NOP + say('Computer wins!\n')
right2 = NOP + say('Tie!\n')

# ---------- layout
TALL = 12      # height of tall number blocks in the main line
WDOWN = 4      # width of number blocks in the downward paths
grid = {}
def put(x, y, col):
    assert (x, y) not in grid, ('overlap', x, y)
    assert x >= 0 and y >= 0
    grid[(x, y)] = col

def nextcol(col, op):
    dh, dl = CMD[op[0]]
    return ((col[0] + dh) % 6, (col[1] + dl) % 3)

def size_for(ops, i):
    """Size of the block reached by op i = value pushed by the *next* op if that is a push, else 1."""
    return ops[i + 1][1] if i + 1 < len(ops) and ops[i + 1][0] == 'push' else 1

def lay_row(ops, col, x, mode, last):
    """Lay ops left to right on row 1. mode 'tall' or 'flat'. Returns (x of last block, colour of last block)."""
    for i, op in enumerate(ops):
        col = nextcol(col, op)
        last_op = i == len(ops) - 1
        if last_op and last == 'terminal':
            for y in range(3): put(x, y, col)               # 3 tall, entered in the middle
            return x, col
        if last_op:                                         # branch block: always 1 codel
            put(x, 1, col); return x, col
        n = size_for(ops, i)
        if mode == 'tall' and n > 1:
            full, rem = divmod(n, TALL)
            cols = full + (1 if rem else 0)
            for c in range(cols):
                h = TALL if c < full else rem
                for r in range(h): put(x + c, 1 + r, col)
            x += cols
        else:
            for c in range(n): put(x + c, 1, col)
            x += n
    raise AssertionError

def lay_down(ops, col, x, y):
    for i, op in enumerate(ops):
        col = nextcol(col, op)
        if i == len(ops) - 1:                               # terminal: 3 wide, entered in the middle
            for dx in (-1, 0, 1): put(x + dx, y, col)
            return
        n = size_for(ops, i)
        if n == 1:
            put(x, y, col); y += 1
        else:
            body = n - 1
            rows = -(-body // WDOWN)
            k = 0
            for r in range(rows):
                for c in range(WDOWN):
                    if k < body: put(x + c, y + r, col); k += 1
            put(x, y + rows, col)                           # single exit codel at the bottom
            y += rows + 1

# B0: an L shape so the main line can run along row 1 (row 0 stays free for the end block)
col0 = (0, 1)
for cell in ((0, 0), (0, 1), (1, 1)): put(cell[0], cell[1], col0)
x1, c1 = lay_row(main, col0, 2, 'tall', 'node')
x2, c2 = lay_row(right1, c1, x1 + 1, 'flat', 'node')
assert x2 - x1 > WDOWN, 'second down-path would overlap the first'
lay_row(right2, c2, x2 + 1, 'flat', 'terminal')
lay_down(down1, c1, x1, 2)
lay_down(down2, c2, x2, 2)

CODEL = int(sys.argv[1]) if len(sys.argv) > 1 else 6
W = max(x for x, _ in grid) + 2
H = max(y for _, y in grid) + 2
img = Image.new('RGB', (W * CODEL, H * CODEL), (0, 0, 0))
px = img.load()
for (x, y), col in grid.items():
    rgb = RGB[col]
    for dy in range(CODEL):
        for dx in range(CODEL): px[x * CODEL + dx, y * CODEL + dy] = rgb
img.save('dice_game.png', optimize=True)
print('image: %d x %d codels (codel size %d px -> %d x %d px); %d ops on the main line' %
      (W, H, CODEL, W * CODEL, H * CODEL, len(main)))

In [ ]:
!python3 piet_gen.py 6

In [ ]:
from PIL import Image
from IPython.display import display
im = Image.open('dice_game.png')
print('full size:', im.size, 'pixels (shown at one quarter size)')
display(im.resize((im.width // 4, im.height // 4), Image.NEAREST))

In [ ]:
!echo 42 | python3 esolang_runners.py piet dice_game.png

Try other seeds, such as `echo 7`, to see all three endings.

## 11. Whitespace

Whitespace (Edwin Brady and Chris Morris, 2003) uses only **space, tab and newline**. Every other character is ignored as a comment, so a Whitespace program can hide inside a file of ordinary text, or look completely blank.

It's a stack machine with a heap. Each instruction is a short pattern of the three characters:

| Group | Prefix | Examples |
|-------|--------|----------|
| Stack | space | push a number, duplicate, swap, discard |
| Arithmetic | tab space | add, subtract, multiply, divide, modulo |
| Heap | tab tab | store, retrieve |
| Flow control | newline | label, call, jump, jump if zero, jump if negative, return, end |
| Input/output | tab newline | print a character or number, read a character or number |

Numbers are written in binary (space = 0, tab = 1) with a sign, ended by a newline.

How the game works:
- **Variables** are numbered heap slots (slot 0 is the seed, slot 1 is your score, and so on).
- **Text** is printed one character at a time (`push 72`, `print character`), which is why the file is large.
- **Random numbers** use the same generator as the Piet version, `seed = (seed + 1) * 75 mod 65537 - 1`, and a die is `seed * 6 / 65536 + 1`.
- **The generator** writes two files: `dice_game.ws` (the real program) and `dice_game.wsa`, a readable listing with one instruction per line.

**Input:** a positive seed number, Enter once per round, then `n` to decline a rematch.

**What we get:** a 7,126-character file of pure whitespace. It plays five rounds and announces a winner, and it can replay.

In [ ]:
%%writefile ws_gen.py
# Generates dice_game.ws (Whitespace) and dice_game.wsa (readable listing)
S, T, L = ' ', '\t', '\n'
ops = []
def op(*a): ops.append(a)
def push(n): op('push', n)
def say(s):
    for ch in s: push(ord(ch)); op('outc')
def get(a): push(a); op('load')
def put(a, v): push(a); push(v); op('store')
def inc(a): push(a); get(a); push(1); op('add'); op('store')
def num(a): get(a); op('outn')
def into(a): push(a); op('swap'); op('store')          # store value on top of stack at heap[a]
def lbl(n): op('label', n)
def call(n): op('call', n)
def jmp(n): op('jump', n)
def jz(n): op('jz', n)
def jneg(n): op('jneg', n)

SEED,PW,CW,RND,P1,P2,C1,C2,PT,CT,CH,ANS = range(12)
NROUNDS = 5

# ---- main program
say("Enter a whole number to seed the dice: ")
push(SEED); op('readn')
push(SEED); get(SEED); push(65536); op('mod'); op('store')

lbl('newgame')
put(PW, 0); put(CW, 0)
say("=== Dice Game: You vs. Computer ===\n")
say("Best of %d rounds; highest total wins.\n" % NROUNDS)
put(RND, 1)

lbl('roundloop')
say("\nRound "); num(RND); say("\n")
say("Press Enter to roll...\n")
call('waitenter')
for a in (P1, P2, C1, C2):
    call('roll'); into(a)
push(PT); get(P1); get(P2); op('add'); op('store')
push(CT); get(C1); get(C2); op('add'); op('store')

say("You rolled:      "); num(P1); say(" + "); num(P2); say(" = "); num(PT); say("\n")
say("Computer rolled: "); num(C1); say(" + "); num(C2); say(" = "); num(CT); say("\n")

get(PT); get(CT); op('sub'); op('dup'); jz('tie'); jneg('cpuwin')
inc(PW); say("You win this round!\n"); jmp('score')
lbl('tie');   op('discard'); say("Tie -- no points.\n"); jmp('score')
lbl('cpuwin'); inc(CW); say("Computer wins this round.\n")

lbl('score')
say("Score: You "); num(PW); say(" - Computer "); num(CW); say("\n")
inc(RND)
get(RND); push(NROUNDS + 1); op('sub'); jneg('roundloop')

say("\n=== Final Result ===\n")
get(PW); get(CW); op('sub'); op('dup'); jz('draw'); jneg('lost')
say("You win the game!\n"); jmp('again')
lbl('draw');  op('discard'); say("The game is a draw.\n"); jmp('again')
lbl('lost');  say("The computer wins the game.\n")

lbl('again')
say("\nPlay again? (y/n)\n")
push(ANS); op('readc')
get(ANS); jneg('quit')
get(ANS); push(10); op('sub'); jz('check')
call('waitenter')
lbl('check')
get(ANS); push(121); op('sub'); jz('newgame')
get(ANS); push(89);  op('sub'); jz('newgame')

lbl('quit')
say("Thanks for playing!\n")
op('end')

# ---- subroutine: wait for Enter (ends the program if input runs out)
lbl('waitenter')
push(CH); op('readc')
get(CH); jneg('quit')
get(CH); push(10); op('sub'); jz('wedone')
jmp('waitenter')
lbl('wedone'); op('ret')

# ---- subroutine: roll one die; leaves 1..6 on the stack.
# seed = ((seed + 1) * 75 mod 65537) - 1  (full-period generator on 0..65535)
# die  = seed * 6 / 65536 + 1
lbl('roll')
get(SEED); push(1); op('add'); push(75); op('mul'); push(65537); op('mod'); push(1); op('sub')
op('dup'); into(SEED)
push(6); op('mul'); push(65536); op('div'); push(1); op('add')
op('ret')

# ---- encode
IMP = {'push':S+S,'dup':S+L+S,'swap':S+L+T,'discard':S+L+L,
       'add':T+S+S+S,'sub':T+S+S+T,'mul':T+S+S+L,'div':T+S+T+S,'mod':T+S+T+T,
       'store':T+T+S,'load':T+T+T,
       'label':L+S+S,'call':L+S+T,'jump':L+S+L,'jz':L+T+S,'jneg':L+T+T,'ret':L+T+L,'end':L+L+L,
       'outc':T+L+S+S,'outn':T+L+S+T,'readc':T+L+T+S,'readn':T+L+T+T}
labels = {}
def lab(name):
    if name not in labels: labels[name] = len(labels) + 1
    return format(labels[name], '08b').replace('0', S).replace('1', T) + L
def number(n):
    return (T if n < 0 else S) + format(abs(n), 'b').replace('0', S).replace('1', T) + L

ws = []
listing = []
for o in ops:
    k = o[0]
    if k == 'push':
        ws.append(IMP[k] + number(o[1])); listing.append('    push %d' % o[1])
    elif k in ('label','call','jump','jz','jneg'):
        ws.append(IMP[k] + lab(o[1]))
        listing.append(('%s:' % o[1]) if k == 'label' else '    %s %s' % (k, o[1]))
    else:
        ws.append(IMP[k]); listing.append('    ' + k)

open('dice_game.ws', 'w').write(''.join(ws))
open('dice_game.wsa', 'w').write('\n'.join(listing) + '\n')
print(len(ops), 'instructions;', len(''.join(ws)), 'bytes')

In [ ]:
!python3 ws_gen.py
!head -12 dice_game.wsa

The listing above is the same program. The next cell shows the real file with whitespace made visible: `^I` is a tab, `$` marks the end of a line, and spaces appear as spaces.

In [ ]:
!head -c 300 dice_game.ws | cat -A

In [ ]:
!printf '42\n\n\n\n\n\nn\n' | python3 esolang_runners.py ws dice_game.ws

## 12. What we ended up with

All eleven versions play the same game, with the simplifications noted in Part 2. Run the cell below to compare them.

In [ ]:
print("Source files (lines of text):")
!wc -l dice_game.py dice_game.f90 dice_game.f dice_game.bas dice_game.c dice_game.s dice_game.hex
print()
print("Esoteric programs (bytes) and the generators that write them (lines):")
!ls -l dice_game.bf dice_game.chef dice_game.png dice_game.ws | awk '{print $5, $9}'
!wc -l bf_gen.py chef_gen.py piet_gen.py ws_gen.py
print()
print("Executables (bytes):")
!ls -l dice_f90 dice_f77 dice_c dice_asm dice_mc | awk '{print $5, $9}'

### How each version got from text to a running program

| Version | Who translated it | Steps we ran |
|---------|-------------------|--------------|
| Python | An interpreter (`python3`) | Run |
| Fortran 90 and 77 | A compiler (`gfortran`) | Compile, then run |
| BASIC | An interpreter (`bwbasic`) | Run |
| C | A compiler and linker (`gcc`) | Compile and link, then run |
| Assembly | An assembler (`as`) and a linker (`ld`) | Assemble, link, then run |
| Machine code | Nobody: we wrote the bytes. `xxd` only turned hex text into a file | Convert, then run |
| Brainfuck, Chef, Piet, Whitespace | An interpreter we wrote in Python (`esolang_runners.py`) | Generate the program, then run |

The pattern: the higher a language sits on the ladder, the more translating happens between your text and the CPU. The lower it sits, the less translating there is, and the more you have to spell out yourself.

### What to take away

| Version | Source you write | What runs | Notes |
|---------|------------------|-----------|-------|
| Python | Shortest and most readable | Interpreted at run time, with no build step | One line to roll a die; the baseline for the rest |
| Fortran 90 | Short and readable | Native executable | Random numbers and loops come built in |
| Fortran 77 | Column rules, labels, `GO TO` | Native executable | You carry your own random generator |
| BASIC | Numbered lines | The interpreter reads your text | No executable; slowest, easiest to start with |
| C | Short, but input handling is manual | Native executable | Linked to the C library, so it is larger than the assembly build |
| Assembly | Longest by far | Small native executable | You write the random generator, number printing and syscalls yourself; the linker pads the file for alignment, so it is bigger than the raw machine code |
| Machine code | None (just bytes) | The same bytes the assembler produces | The smallest file; unreadable without a disassembler |
| Brainfuck | A generator emits 26,630 commands | Any Brainfuck interpreter | Eight commands; one game, no rematch |
| Chef | A generator emits a 52 KB recipe | A Chef interpreter | Dice come from shuffling a bowl; one game, no rematch |
| Piet | A generator draws the image | A Piet interpreter | Numbers are block sizes; one round only |
| Whitespace | A generator emits 7,126 characters | A Whitespace interpreter | Full game with rematch; the file looks blank |

Four ideas worth remembering:

1. **Every step down trades convenience for control.** The game logic never changes: roll, add, compare, update the score. What changes is how much work you do by hand to express it.
2. **The compiler and the assembler are doing the same job at different levels.** Fortran, C, and BASIC all end up as machine code (or, for BASIC, as work the interpreter does for you). The disassembly above shows the exact bytes that become the instructions you wrote.
3. **The random numbers differ in every version.** Each one uses a different generator (Fortran's built-in, Park-Miller, BASIC's `RND`, C's `rand`, xorshift64, a bowl shuffle in Chef, and the small generator in Piet and Whitespace), and some are seeded from the clock. Even with the same seed, the dice won't match across languages.
4. **Computation needs very little.** Brainfuck has eight commands, Piet has no text, Whitespace has no visible characters, and Chef has no `if`. All of them still ran the game. What a language really gives you is not capability but how much work it takes to say something.

### Where to go next

- Change the number of rounds (`NROUND` / `n_rounds` / `N_ROUNDS` / `N`) and rebuild each version. In the assembly and hex versions the "Best of 5" text is hardcoded, so you would also change that message.
- Add a rule, such as doubles winning outright, and see how much code each language needs.
- Go one level below machine code: write the game as hardware in Verilog, with a linear-feedback shift register as the dice and a comparator for the winner.
- Open `bf_gen.py` and change `rounds`, or edit the strings the generators print, then rerun the generator and the program.
- Try the generated `.bf`, `.chef`, `.ws` files and the Piet image in an interpreter written by someone else, and see whether they agree with the ones in this notebook.